# ColBERT Late-Interaction Search with GEM Native Index

This notebook demonstrates voyager-index's native multi-vector search capabilities
using synthetic ColBERT-style embeddings and the GEM graph index.

**What you'll learn:**
- Create a GEM-powered multi-vector index
- Add documents with per-token embeddings
- Search with multi-vector queries
- Use payload filtering
- CRUD operations (update, delete)
- Understand quantization and qCH scoring
- Benchmark GEM search performance

In [ ]:
import voyager_index
import numpy as np
import time
import tempfile
from pathlib import Path

print(f"voyager-index version: {voyager_index.__version__}")

## 1. Create a GEM Native Index

The `voyager_index.Index` class automatically selects the best backend.
With `latence-gem-index` installed, it uses the native GEM graph.

In [ ]:
DIM = 128  # ColBERT embedding dimension
N_DOCS = 500
TOKENS_PER_DOC = 64  # average tokens per document

index_path = tempfile.mkdtemp(prefix="voyager_colbert_")
index = voyager_index.Index(
    index_path,
    dim=DIM,
    engine="gem",
    n_fine=64,
    n_coarse=8,
    max_degree=32,
    ef_construction=100,
    seed_batch_size=64,
)
print(index)

## 2. Generate Synthetic ColBERT Embeddings

In a real pipeline, these would come from a ColBERT model.
Here we create topically clustered documents to simulate real data.

In [ ]:
rng = np.random.RandomState(42)
N_TOPICS = 10
topic_centers = rng.randn(N_TOPICS, DIM).astype(np.float32)

categories = ["science", "technology", "history", "art", "sports",
              "politics", "health", "finance", "education", "nature"]

documents = []
payloads = []
for i in range(N_DOCS):
    topic = i % N_TOPICS
    n_tokens = rng.randint(32, TOKENS_PER_DOC + 32)
    noise = rng.randn(n_tokens, DIM).astype(np.float32) * 0.3
    doc_vecs = topic_centers[topic] + noise
    documents.append(doc_vecs)
    payloads.append({
        "category": categories[topic],
        "word_count": n_tokens * 5,
        "doc_idx": i,
    })

print(f"Generated {len(documents)} documents")
print(f"Example shapes: {[d.shape for d in documents[:3]]}")

## 3. Index Documents

In [ ]:
t0 = time.perf_counter()
ids = index.add(documents, payloads=payloads)
build_time = time.perf_counter() - t0

print(f"Indexed {len(ids)} documents in {build_time:.2f}s")
print(f"IDs: {ids[:10]}...")
print(f"\nIndex stats:")
stats = index.stats()
print(f"  Engine: {stats.engine}")
print(f"  Total vectors: {stats.total_vectors}")
print(f"  Active vectors: {stats.active_vectors}")

## 4. Multi-Vector Search

In [ ]:
# Simulate a ColBERT query (multiple token vectors)
query_topic = 0  # "science"
query = topic_centers[query_topic] + rng.randn(5, DIM).astype(np.float32) * 0.2

t0 = time.perf_counter()
results = index.search(query, k=10)
search_time = (time.perf_counter() - t0) * 1000

print(f"Search completed in {search_time:.2f}ms\n")
print(f"{'Rank':<6} {'ID':<6} {'Score':<10} {'Category':<12} {'Words':<8}")
print("-" * 45)
for i, r in enumerate(results):
    print(f"{i+1:<6} {r.id:<6} {r.score:<10.4f} {r.payload['category']:<12} {r.payload['word_count']:<8}")

## 5. Payload Filtering

Search within a specific category or word count range.

In [ ]:
# Only science documents
results = index.search(query, k=5, filters={"category": "science"})
print("Filter: category = 'science'")
for r in results:
    print(f"  ID={r.id}, score={r.score:.4f}, category={r.payload['category']}")

print()

# Not-equal filter
results = index.search(query, k=5, filters={"category": {"$ne": "science"}})
print("Filter: category != 'science'")
for r in results:
    print(f"  ID={r.id}, score={r.score:.4f}, category={r.payload['category']}")

print()

# Range filter
results = index.search(query, k=5, filters={"word_count": {"$gte": 200, "$lte": 300}})
print("Filter: 200 <= word_count <= 300")
for r in results:
    print(f"  ID={r.id}, score={r.score:.4f}, words={r.payload['word_count']}")

## 6. CRUD Operations

In [ ]:
# Update a payload
index.update_payload(ids[0], {"category": "updated_science", "priority": "high"})
retrieved = index.get(ids[0])
print(f"After update: {retrieved}")

# Delete documents
index.delete([ids[0], ids[1]])
results_after = index.search(query, k=5)
deleted_ids = {ids[0], ids[1]}
assert all(r.id not in deleted_ids for r in results_after)
print(f"\nDeleted {ids[0]} and {ids[1]} — verified not in search results")

# Scroll through documents
page = index.scroll(limit=5, offset=0)
print(f"\nScroll page: {len(page.results)} results, next_offset={page.next_offset}")
for doc in page.results:
    print(f"  ID={doc['id']}, category={doc['payload'].get('category', 'N/A')}")

## 7. Performance Benchmark: GEM vs Brute-Force

In [ ]:
n_queries = 100
gem_times = []
bf_times = []

for _ in range(n_queries):
    q = topic_centers[rng.randint(0, N_TOPICS)] + rng.randn(4, DIM).astype(np.float32) * 0.2
    
    t0 = time.perf_counter()
    _ = index.search(q, k=10)
    gem_times.append((time.perf_counter() - t0) * 1000)
    
    # Brute-force MaxSim
    t0 = time.perf_counter()
    scores = []
    for doc in documents[2:]:  # skip deleted
        sim = q @ doc.T
        scores.append(sim.max(axis=1).sum())
    bf_times.append((time.perf_counter() - t0) * 1000)

gem_p50 = sorted(gem_times)[len(gem_times) // 2]
bf_p50 = sorted(bf_times)[len(bf_times) // 2]

print(f"GEM Search:    p50 = {gem_p50:.2f}ms")
print(f"Brute-Force:   p50 = {bf_p50:.2f}ms")
print(f"Speedup:       {bf_p50 / gem_p50:.1f}x")

In [ ]:
# Snapshot for backup
snap = index.snapshot()
print(f"Snapshot saved: {snap}")
print(f"Size: {snap.stat().st_size / 1024:.1f} KB")

index.close()
print("\nIndex closed. Done!")